In [11]:
!pip install -q --upgrade gradio

In [13]:

import gradio as gr
import random
import math


EMPTY = ""
HUMAN = "X"
AI = "O"

WINNING_LINES = [
    (0, 1, 2), (3, 4, 5), (6, 7, 8),
    (0, 3, 6), (1, 4, 7), (2, 5, 8),
    (0, 4, 8), (2, 4, 6)
]

def check_winner(board):

    for a, b, c in WINNING_LINES:
        if board[a] == board[b] == board[c] != EMPTY:
            return board[a]

    if EMPTY not in board:
        return "Draw"

    return None

def minimax(board, depth, maximizing, alpha, beta):

    result = check_winner(board)

    if result == AI:
        return 10 - depth

    if result == HUMAN:
        return depth - 10

    if result == "Draw":
        return 0

    if maximizing:

        best = -math.inf

        for i in range(9):

            if board[i] == EMPTY:

                board[i] = AI

                score = minimax(
                    board, depth + 1,
                    False, alpha, beta
                )

                board[i] = EMPTY

                best = max(best, score)
                alpha = max(alpha, best)

                if beta <= alpha:
                    break

        return best

    else:

        best = math.inf

        for i in range(9):

            if board[i] == EMPTY:

                board[i] = HUMAN

                score = minimax(
                    board, depth + 1,
                    True, alpha, beta
                )

                board[i] = EMPTY

                best = min(best, score)
                beta = min(beta, best)

                if beta <= alpha:
                    break

        return best

def get_ai_move(board, difficulty):

    available = [
        i for i in range(9)
        if board[i] == EMPTY
    ]

    if not available:
        return None

    if difficulty == "Easy":
        return random.choice(available)

    if difficulty == "Medium":
        if random.random() < 0.35:
            return random.choice(available)

    best_score = -math.inf
    best_moves = []

    for move in available:

        board[move] = AI

        score = minimax(
            board, 0, False,
            -math.inf, math.inf
        )

        board[move] = EMPTY

        if score > best_score:
            best_score = score
            best_moves = [move]

        elif score == best_score:
            best_moves.append(move)

    return random.choice(best_moves)

def new_state():

    return {
        "board": [EMPTY] * 9,
        "player_wins": 0,
        "ai_wins": 0,
        "draws": 0,
        "finished": False
    }

def render_board(board, finished=False):

    updates = []

    for i in range(9):

        value = board[i] if board[i] else str(i + 1)

        updates.append(
            gr.update(
                value=value,
                interactive=(
                    board[i] == EMPTY and not finished
                )
            )
        )

    return updates

def score_text(state):

    return (
        f"👤 Player: {state['player_wins']}   |   "
        f"🤖 AI: {state['ai_wins']}   |   "
        f"🤝 Draws: {state['draws']}"
    )

def play(position, state, difficulty):

    state = dict(state)
    board = state["board"].copy()

    if state["finished"] or board[position] != EMPTY:

        return (
            state,
            "⚠️ Invalid move! Choose another box.",
            score_text(state),
            *render_board(board, state["finished"])
        )

    board[position] = HUMAN

    result = check_winner(board)

    if result == HUMAN:

        state["player_wins"] += 1
        state["finished"] = True
        state["board"] = board

        return (
            state,
            "🎉 Congratulations! You Won!",
            score_text(state),
            *render_board(board, True)
        )

    if result == "Draw":

        state["draws"] += 1
        state["finished"] = True
        state["board"] = board

        return (
            state,
            "🤝 Match Draw!",
            score_text(state),
            *render_board(board, True)
        )

    move = get_ai_move(board, difficulty)

    if move is not None:
        board[move] = AI

    result = check_winner(board)

    if result == AI:

        state["ai_wins"] += 1
        state["finished"] = True
        message = "🤖 AI Wins! Better luck next time!"

    elif result == "Draw":

        state["draws"] += 1
        state["finished"] = True
        message = "🤝 Match Draw!"

    else:

        message = "🎮 Your Turn! Select another box."

    state["board"] = board

    return (
        state,
        message,
        score_text(state),
        *render_board(board, state["finished"])
    )

def reset_game(state, first_player):

    state = dict(state)
    board = [EMPTY] * 9

    state["board"] = board
    state["finished"] = False

    if first_player == "AI":

        move = get_ai_move(board, "Hard")
        board[move] = AI
        state["board"] = board

        message = "🤖 AI starts! Your turn now."

    else:

        message = "🎮 New Game! You start first."

    return (
        state,
        message,
        score_text(state),
        *render_board(board)
    )


CSS = """
.game-cell button {
    min-height: 100px !important;
    font-size: 38px !important;
    font-weight: bold !important;
    border-radius: 15px !important;
}
"""

with gr.Blocks(
    theme=gr.themes.Soft(),
    css=CSS,
    title="Advanced Tic Tac Toe AI"
) as demo:

    gr.Markdown("""
    # 🎮 ADVANCED TIC TAC TOE AI

    ### Challenge the Intelligent Computer!

    **YOU = X  |  AI = O**

    Select any numbered box to make your move.
    """)

    game_state = gr.State(new_state())

    with gr.Row():

        difficulty = gr.Dropdown(
            choices=["Easy", "Medium", "Hard"],
            value="Hard",
            label="🎯 AI Difficulty",
            scale=1
        )

        first_player = gr.Dropdown(
            choices=["Human", "AI"],
            value="Human",
            label="🚀 First Move",
            scale=1
        )

    status = gr.Textbox(
        value="Welcome! Start playing.",
        label="📢 Game Status",
        interactive=False
    )

    scoreboard = gr.Textbox(
        value="Player: 0 | AI: 0 | Draws: 0",
        label="🏆 Scoreboard",
        interactive=False
    )

    gr.Markdown("## 🎲 GAME BOARD")

    buttons = []

    for row in range(3):

        with gr.Row(equal_height=True):

            for col in range(3):

                position = row * 3 + col

                button = gr.Button(
                    value=str(position + 1),
                    size="lg",
                    min_width=100,
                    elem_classes=["game-cell"]
                )

                buttons.append(button)

    with gr.Row():

        restart = gr.Button(
            "🔄 Restart Game",
            variant="primary"
        )

    gr.Markdown("""
    ## ⭐ FEATURES

    - Interactive 3x3 Game Board
    - Human vs AI Gameplay
    - Minimax Algorithm
    - Alpha-Beta Pruning
    - Easy, Medium and Hard Levels
    - Automatic Win Detection
    - Live Scoreboard
    - Restart Game Option

    **Developed for CodSoft Python Internship - Task 2**
    """)

    for i, button in enumerate(buttons):

        button.click(
            fn=lambda state, diff, p=i: play(
                p, state, diff
            ),
            inputs=[game_state, difficulty],
            outputs=[
                game_state,
                status,
                scoreboard,
                *buttons
            ]
        )

    restart.click(
        fn=reset_game,
        inputs=[game_state, first_player],
        outputs=[
            game_state,
            status,
            scoreboard,
            *buttons
        ]
    )

demo.launch(share=True)

/tmp/ipykernel_1243/431748660.py:280: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme, css. Please pass these parameters to launch() instead.
  with gr.Blocks(


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://aae463a2f20fe61e25.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
